<a href="https://colab.research.google.com/github/adib422/FlyRank-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("HF Token: ")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Feature Vector

The feature vector is built using only information available before the prediction point.

Features:

Previous 30-day impressions (imp_prev30)
Previous 30-day clicks (clk_prev30)
Average search position (pos_last30)
Visible query count
Top query share

Missing values are handled by removing rows with missing feature values before model training. Client and content identifiers are used only for joins and grouping, not as model features.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_vector = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM {TABLES['fact_daily']}
),

base AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date <= end_d - INTERVAL 30 DAY
                THEN gsc_impressions
                ELSE 0
            END
        ) AS imp_prev30,

        SUM(
            CASE
                WHEN report_date <= end_d - INTERVAL 30 DAY
                THEN gsc_clicks
                ELSE 0
            END
        ) AS clk_prev30,

        AVG(
            CASE
                WHEN report_date > end_d - INTERVAL 30 DAY
                THEN gsc_avg_position
            END
        ) AS pos_last30

    FROM {TABLES['fact_daily']}, bounds

    WHERE report_date > end_d - INTERVAL 60 DAY

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM base
LIMIT 10
""").df()

feature_vector


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,imp_prev30,clk_prev30,pos_last30
0,client_3ffa76342f366962,content_de1e22d87ebde3a9,0.0,0.0,104.000000
1,client_3ffa76342f366962,content_5d3e741a3dd59f22,0.0,0.0,NaN
2,client_3ffa76342f366962,content_d4c3d32484f3e5ff,0.0,0.0,NaN
3,client_3ffa76342f366962,content_92de78ea4583fc68,1.0,0.0,10.222222
4,client_3ffa76342f366962,content_ff0afc801044c944,0.0,0.0,NaN
5,client_3ffa76342f366962,content_ab8171c9576be3ba,0.0,0.0,0.000000
6,client_3ffa76342f366962,content_d8e9c5058ab5c50f,1.0,0.0,NaN
7,client_3ffa76342f366962,content_0a0550ad707e8667,0.0,0.0,NaN
8,client_3ffa76342f366962,content_2525403002e456ee,0.0,0.0,NaN
9,client_3ffa76342f366962,content_046287de987107e3,0.0,0.0,4.791667


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## Feature Notes

The feature vector uses only historical information that is available before the prediction point.

| Feature | Meaning | Missing Value Handling | Available Before Prediction |
|----------|----------|------------------------|-----------------------------|
| `imp_prev30` | Total impressions during the previous 30-day window | Rows with missing values are excluded | Yes |
| `clk_prev30` | Total clicks during the previous 30-day window | Rows with missing values are excluded | Yes |
| `pos_last30` | Average Google Search position over the observation window | DuckDB ignores NULL values during aggregation | Yes |
| `visible_queries` | Number of visible search queries for a page | Missing rows are excluded during modeling | Yes |
| `top_query_share` | Fraction of impressions generated by the top query | Missing rows are excluded during modeling | Yes |

### Missing Values

The selected features contain very few missing values. Missing records are removed before training rather than being filled with artificial values.

### Categorical Features

This feature vector contains only numerical features. The identifier columns (`client_hash_id` and `content_hash_id`) are used only for joins and grouping and are **not** provided to the machine learning model.

### Availability

Every feature is computed using historical observations that exist **before** the prediction window. No future information is used when constructing the feature vector.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_vector.info()
feature_vector.describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   client_hash_id   10 non-null     object 
 1   content_hash_id  10 non-null     object 
 2   imp_prev30       10 non-null     float64
 3   clk_prev30       10 non-null     float64
 4   pos_last30       4 non-null      float64
dtypes: float64(3), object(2)
memory usage: 532.0+ bytes


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,pos_last30
count,10,10,10.000000,10.0,4.000000
unique,1,10,NaN,NaN,NaN
top,client_3ffa76342f366962,content_de1e22d87ebde3a9,NaN,NaN,NaN
freq,10,1,NaN,NaN,NaN
mean,NaN,NaN,0.200000,0.0,29.753472
std,NaN,NaN,0.421637,0.0,49.673526
min,NaN,NaN,0.000000,0.0,0.000000
25%,NaN,NaN,0.000000,0.0,3.593750
50%,NaN,NaN,0.000000,0.0,7.506944
75%,NaN,NaN,0.000000,0.0,33.666667


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Leakage Hunt**

The feature vector was reviewed for possible leakage.

Checks performed:

No future impressions used.

No future clicks used.

No future dates used.

No label-derived columns included.

Client identifiers are not model features.

Content identifiers are not model features.

Only historical information is used for prediction.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
candidate_features = [
    "client_hash_id",
    "content_hash_id",
    "imp_prev30",
    "clk_prev30",
    "pos_last30"
]

print("Candidate Features")
print(candidate_features)

leakage = []

for col in candidate_features:
    lower = col.lower()

    if "future" in lower:
        leakage.append(col)

    if "label" in lower:
        leakage.append(col)

    if "declining" in lower:
        leakage.append(col)

print()

if leakage:
    print("Leakage Found")
    print(leakage)
else:
    print("No leakage detected.")

Candidate Features
['client_hash_id', 'content_hash_id', 'imp_prev30', 'clk_prev30', 'pos_last30']

No leakage detected.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## Excluded Fields

The following fields were intentionally excluded from the feature vector to prevent target leakage or because they are used only as context.

| Excluded Field | Reason |
|----------------|--------|
| `client_hash_id` | Used only for grouping and joins, not as a predictive feature |
| `content_hash_id` | Used only for grouping and joins, not as a predictive feature |
| `report_date` | Defines the observation window and should not be learned directly |
| Future impressions | Would leak information from the prediction period |
| Future clicks | Would leak information from the prediction period |
| Label columns | Directly reveal the prediction target |
| Client-specific identifiers | Context only, not meaningful predictive signals |

### Leakage Prevention

Only historical measurements available before the prediction point are included in the feature vector. Any variable that depends on future outcomes or directly reveals the prediction target is excluded.

### Privacy

The dataset uses pseudonymized identifiers. No client names, URLs, or personally identifiable information are used in the feature vector.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded = pd.DataFrame({
    "Excluded_Field": [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "future_impressions",
        "future_clicks",
        "label"
    ],
    "Reason": [
        "Grouping only",
        "Grouping only",
        "Window definition",
        "Future information",
        "Future information",
        "Target leakage"
    ]
})

excluded

,Excluded_Field,Reason
0,client_hash_id,Grouping only
1,content_hash_id,Grouping only
2,report_date,Window definition
3,future_impressions,Future information
4,future_clicks,Future information
5,label,Target leakage


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.